# EDA — Stuff+ LMB (pitch tracking data)

Exploración inicial del dataset `data/raw/stuff_model_df.parquet` (635,002 lanzamientos,
84 columnas) para entender la distribución de los datos antes de modelar Stuff+, con
foco especial en el efecto de la altitud (`altitude_category`).

**Nota sobre altitud:** el dataset no incluye un nombre de estadio explícito, solo el
bucket `altitude_category` con tres valores: `No Altitude`, `Medium Altitude` y
`Extreme Altitude`. Dado que el Estadio Alfredo Harp Helú (CDMX, ~2,240 msnm) es por
mucho el parque de mayor altitud de la liga, en este notebook **asumimos que
`Extreme Altitude` corresponde al Harp Helú**. Esta asunción debe confirmarse con el
equipo de datos de Diablos antes de usarse en el modelo final.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "stuff_model_df.parquet"
FIG_DIR = ROOT / "reports" / "figures"
REPORT_DIR = ROOT / "reports"
FIG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["RelSpeed", "SpinRate", "InducedVertBreak", "HorzBreak", "Extension", "RelHeight"]
HARP_LABEL = "Extreme Altitude"  # ver nota arriba

DATA_PATH


PosixPath('/media/mrfett/A002/Hackathon_diablos/data/raw/stuff_model_df.parquet')

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f"shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head(3)


shape: 635,002 filas x 84 columnas


,year,PitchUID,game_anon_id,pitcher_anon_id,batter_anon_id,catcher_anon_id,PitcherThrows,BatterSide,altitude_category,AutoPitchType,...,Top/Bottom,Outs,Balls,Strikes,count,PlateLocHeight,PlateLocSide,in_strike_zone,outside_strike_zone,swung_outside_strike_zone
0,2024,d6a82560-f939-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Sinker,...,Top,0.0,0.0,0.0,0-0,4.19842,-1.06959,0,1.0,0.0
1,2024,df15bd20-f939-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Sinker,...,Top,0.0,1.0,0.0,1-0,1.75154,-0.20261,1,0.0,0.0
2,2024,e965d080-f939-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Cutter,...,Top,0.0,1.0,1.0,1-1,2.05249,1.41929,0,1.0,0.0


## 1. Distribución de pitches por year, altitude_category y AutoPitchType

In [3]:
year_counts = df["year"].value_counts().sort_index()
alt_counts = df["altitude_category"].value_counts(dropna=False).reindex(
    ["No Altitude", "Medium Altitude", "Extreme Altitude", np.nan]
)
pitchtype_counts = df["AutoPitchType"].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.barplot(x=year_counts.index.astype(str), y=year_counts.values, ax=axes[0], color="#4C72B0")
axes[0].set_title("Pitches por year")
axes[0].set_xlabel("year")
axes[0].set_ylabel("n pitches")
for i, v in enumerate(year_counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)

alt_plot = alt_counts.dropna()
sns.barplot(x=alt_plot.index, y=alt_plot.values, ax=axes[1], color="#DD8452")
axes[1].set_title("Pitches por altitude_category")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)
for i, v in enumerate(alt_plot.values):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
n_missing_alt = df["altitude_category"].isna().sum()
axes[1].set_xlabel(f"({n_missing_alt} pitches sin altitude_category, no graficados)")

pt_plot = pitchtype_counts[pitchtype_counts >= 50]  # oculta clases residuales (n<50)
sns.barplot(x=pt_plot.values, y=pt_plot.index, ax=axes[2], color="#55A868")
axes[2].set_title("Pitches por AutoPitchType (n>=50)")
axes[2].set_xlabel("n pitches")

plt.tight_layout()
fig.savefig(FIG_DIR / "01_distribucion_year_altitud_pitchtype.png", bbox_inches="tight")
plt.show()

print("year_counts:\n", year_counts, "\n")
print("altitude_category counts:\n", df["altitude_category"].value_counts(dropna=False), "\n")
print("AutoPitchType counts:\n", pitchtype_counts)


year_counts:
 year
2024    243254
2025    256144
2026    135604
Name: count, dtype: int64 

altitude_category counts:
 altitude_category
No Altitude         293875
Extreme Altitude    183267
Medium Altitude     154840
NaN                   3020
Name: count, dtype: int64 

AutoPitchType counts:
 AutoPitchType
Four-Seam          217618
Slider             139129
Sinker             110897
Changeup            78925
Curveball           54024
Cutter              26468
Splitter             7289
Sweeper               588
Other                  61
OneSeamFastBall         1
TwoSeamFastBall         1
Knuckleball             1
Name: count, dtype: int64


/tmp/ipykernel_13815/4053686418.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Estadísticas descriptivas por altitude_category y por AutoPitchType

Media, desviación estándar y percentiles 1/25/50/75/99 de `RelSpeed`, `SpinRate`,
`InducedVertBreak`, `HorzBreak`, `Extension` y `RelHeight`.


In [4]:
def descriptive_stats(group_df):
    out = {}
    for m in METRICS:
        s = group_df[m].dropna()
        out[(m, "mean")] = s.mean()
        out[(m, "std")] = s.std()
        out[(m, "p1")] = s.quantile(0.01)
        out[(m, "p25")] = s.quantile(0.25)
        out[(m, "p50")] = s.quantile(0.50)
        out[(m, "p75")] = s.quantile(0.75)
        out[(m, "p99")] = s.quantile(0.99)
        out[(m, "n")] = s.shape[0]
    return pd.Series(out)

stats_by_altitude = (
    df.groupby("altitude_category", dropna=False)
    .apply(descriptive_stats, include_groups=False)
    .round(3)
)
stats_by_pitchtype = (
    df.groupby("AutoPitchType", dropna=False)
    .apply(descriptive_stats, include_groups=False)
    .round(3)
)

stats_by_altitude.to_csv(REPORT_DIR / "desc_stats_by_altitude.csv")
stats_by_pitchtype.to_csv(REPORT_DIR / "desc_stats_by_pitchtype.csv")

print("Guardado: reports/desc_stats_by_altitude.csv y reports/desc_stats_by_pitchtype.csv")
stats_by_altitude


Guardado: reports/desc_stats_by_altitude.csv y reports/desc_stats_by_pitchtype.csv


RelSpeed                                                 \
                      mean    std      p1     p25     p50     p75     p99   
altitude_category                                                           
Extreme Altitude    87.622  5.512  73.797  83.703  88.563  92.020  96.821   
Medium Altitude     87.463  5.431  73.945  83.546  88.477  91.746  96.625   
No Altitude         87.260  5.635  72.829  83.306  88.227  91.682  96.661   
NaN                 84.714  6.038  69.165  80.603  85.376  89.270  96.455   

                             SpinRate           ... Extension            \
                          n      mean      std  ...       p99         n   
altitude_category                               ...                       
Extreme Altitude   183267.0  2292.037  318.385  ...     7.204  182994.0   
Medium Altitude    154840.0  2297.312  333.482  ...     7.311  154618.0   
No Altitude        293875.0  2254.983  364.176  ...     7.168  293598.0   
NaN                  3020.0  2207.776  281.134  ...     6.966    3019.0   

                  RelHeight                                            \
                       mean    std     p1    p25    p50    p75    p99   
altitude_category                                                       
Extreme Altitude      5.752  0.484  4.417  5.446  5.785  6.076  6.771   
Medium Altitude       5.791  0.477  4.511  5.500  5.795  6.110  6.801   
No Altitude           5.767  0.505  4.285  5.478  5.813  6.098  6.800   
NaN                   5.686  0.408  4.603  5.430  5.701  5.963  6.551   

                             
                          n  
altitude_category            
Extreme Altitude   183267.0  
Medium Altitude    154840.0  
No Altitude        293875.0  
NaN                  3020.0  

[4 rows x 48 columns]

In [5]:
stats_by_pitchtype


RelSpeed                                                 \
                    mean    std      p1     p25     p50     p75     p99   
AutoPitchType                                                             
Changeup          83.193  3.529  73.944  81.020  83.532  85.604  91.744   
Curveball         77.728  3.626  67.542  75.665  78.055  80.138  84.783   
Cutter            88.569  2.246  83.249  87.085  88.549  90.018  94.052   
Four-Seam         91.757  2.649  85.523  90.001  91.852  93.581  97.591   
Other             63.813  4.437  53.710  62.799  63.821  64.546  82.877   
Sinker            91.143  2.584  85.146  89.409  91.152  92.923  96.834   
Slider            83.755  3.005  76.618  81.744  83.861  85.848  90.240   
Splitter          83.851  3.363  76.433  81.536  83.430  86.071  91.692   
OneSeamFastBall   76.383    NaN  76.383  76.383  76.383  76.383  76.383   
Sweeper           80.478  3.478  72.335  78.214  80.257  83.127  87.054   
TwoSeamFastBall   93.872    NaN  93.872  93.872  93.872  93.872  93.872   
Knuckleball       83.385    NaN  83.385  83.385  83.385  83.385  83.385   

                           SpinRate           ... Extension            \
                        n      mean      std  ...       p99         n   
AutoPitchType                                 ...                       
Changeup          78925.0  1810.235  333.232  ...     7.304   78826.0   
Curveball         54024.0  2528.691  318.575  ...     6.684   53999.0   
Cutter            26468.0  2391.079  211.417  ...     7.216   26453.0   
Four-Seam        217618.0  2324.216  175.121  ...     7.347  217336.0   
Other                61.0  2211.064   95.195  ...     6.498      61.0   
Sinker           110897.0  2253.592  170.313  ...     7.192  110670.0   
Slider           139129.0  2421.356  330.774  ...     6.904  139013.0   
Splitter           7289.0  1115.093  234.361  ...     6.936    7280.0   
OneSeamFastBall       1.0  1809.717      NaN  ...     5.295       1.0   
Sweeper             588.0  2541.065  255.681  ...     6.586     588.0   
TwoSeamFastBall       1.0  2098.153      NaN  ...     5.723       1.0   
Knuckleball           1.0  2609.483      NaN  ...     5.514       1.0   

                RelHeight                                                      
                     mean    std     p1    p25    p50    p75    p99         n  
AutoPitchType                                                                  
Changeup            5.675  0.549  3.855  5.369  5.729  6.040  6.742   78925.0  
Curveball           5.771  0.516  4.151  5.487  5.812  6.103  6.782   54024.0  
Cutter              5.902  0.405  4.941  5.628  5.905  6.172  6.823   26468.0  
Four-Seam           5.921  0.406  5.009  5.650  5.924  6.191  6.874  217618.0  
Other               5.569  0.435  4.982  5.156  5.355  5.998  6.360      61.0  
Sinker              5.545  0.514  3.812  5.262  5.575  5.892  6.547  110897.0  
Slider              5.732  0.476  4.456  5.432  5.753  6.057  6.756  139129.0  
Splitter            5.796  0.543  4.661  5.458  5.874  6.171  6.926    7289.0  
OneSeamFastBall     5.097    NaN  5.097  5.097  5.097  5.097  5.097       1.0  
Sweeper             5.392  0.612  3.201  5.052  5.464  5.820  6.402     588.0  
TwoSeamFastBall     5.963    NaN  5.963  5.963  5.963  5.963  5.963       1.0  
Knuckleball         5.786    NaN  5.786  5.786  5.786  5.786  5.786       1.0  

[12 rows x 48 columns]

## 3. Boxplots: InducedVertBreak y HorzBreak — Harp Helú vs resto de estadios, por AutoPitchType

Se compara el bucket `Extreme Altitude` (asumido Harp Helú) contra el resto de
altitude_category combinado (`No Altitude` + `Medium Altitude`), para cada tipo de pitch
con muestra suficiente (n >= 200 en ambos grupos).


In [6]:
df_box = df.dropna(subset=["altitude_category"]).copy()
df_box["altitude_group"] = np.where(
    df_box["altitude_category"] == HARP_LABEL, "Harp Helu (Extreme Altitude)", "Resto de estadios"
)

# tipos de pitch con muestra suficiente en ambos grupos
counts_by_group = df_box.groupby(["AutoPitchType", "altitude_group"]).size().unstack(fill_value=0)
valid_pitch_types = counts_by_group[(counts_by_group.min(axis=1) >= 200)].index.tolist()
df_box_valid = df_box[df_box["AutoPitchType"].isin(valid_pitch_types)]
order = (
    df_box_valid.groupby("AutoPitchType").size().sort_values(ascending=False).index.tolist()
)

for metric, fname in [
    ("InducedVertBreak", "02_box_inducedvertbreak_harp_vs_resto.png"),
    ("HorzBreak", "03_box_horzbreak_harp_vs_resto.png"),
]:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(
        data=df_box_valid, x="AutoPitchType", y=metric, hue="altitude_group",
        order=order, showfliers=False, ax=ax,
    )
    ax.set_title(f"{metric}: Harp Helu vs resto de estadios, por AutoPitchType")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title="")
    plt.tight_layout()
    fig.savefig(FIG_DIR / fname, bbox_inches="tight")
    plt.show()

print("Tipos de pitch incluidos (n>=200 en ambos grupos):", order)


/tmp/ipykernel_13815/4184334498.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Tipos de pitch incluidos (n>=200 en ambos grupos): ['Four-Seam', 'Slider', 'Sinker', 'Changeup', 'Curveball', 'Cutter', 'Splitter']


/tmp/ipykernel_13815/4184334498.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Muestra por pitcher: pitchers únicos y distribución de pitches por pitcher

Relevante para decidir si se necesita *shrinkage* bayesiano hacia la media de la liga
para pitchers con pocas observaciones.


In [7]:
pitcher_counts = df.groupby("pitcher_anon_id").size().sort_values(ascending=False)
n_unique_pitchers = pitcher_counts.shape[0]

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(pitcher_counts.values, bins=50, ax=ax, color="#4C72B0")
ax.set_title(f"Distribucion de pitches por pitcher (n={n_unique_pitchers} pitchers unicos)")
ax.set_xlabel("n pitches por pitcher")
ax.set_ylabel("n pitchers")
for q, color in [(0.25, "orange"), (0.5, "red"), (0.75, "green")]:
    val = pitcher_counts.quantile(q)
    ax.axvline(val, color=color, linestyle="--", linewidth=1, label=f"p{int(q*100)}={val:.0f}")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "04_pitches_por_pitcher_hist.png", bbox_inches="tight")
plt.show()

pitcher_desc = pitcher_counts.describe(percentiles=[0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
print(f"n pitchers unicos: {n_unique_pitchers}")
print(pitcher_desc)

n_low_sample = (pitcher_counts < 100).sum()
pct_low_sample = 100 * n_low_sample / n_unique_pitchers
print(f"\nPitchers con < 100 pitches: {n_low_sample} ({pct_low_sample:.1f}% del total)")


n pitchers unicos: 1134
count    1134.000000
mean      557.081129
std       643.921052
min         1.000000
1%          8.000000
10%        22.000000
25%        80.000000
50%       331.000000
75%       770.750000
90%      1375.700000
99%      2988.910000
max      4222.000000
dtype: float64

Pitchers con < 100 pitches: 309 (27.2% del total)


/tmp/ipykernel_13815/860353936.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. % de valores en PitchCall, hit_type y play_result

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

pct_tables = {}
for ax, col in zip(axes, ["PitchCall", "hit_type", "play_result"]):
    pct = (df[col].value_counts(dropna=False, normalize=True) * 100).round(2)
    pct.index = pct.index.astype(str).where(pct.index.notna(), "NaN") if hasattr(pct.index, "notna") else pct.index
    pct_tables[col] = pct
    sns.barplot(x=pct.values, y=pct.index.astype(str), ax=ax, color="#8172B2")
    ax.set_title(col)
    ax.set_xlabel("% de pitches")

plt.tight_layout()
fig.savefig(FIG_DIR / "05_pct_pitchcall_hittype_playresult.png", bbox_inches="tight")
plt.show()

for col, pct in pct_tables.items():
    print(f"--- {col} (%) ---")
    print(pct)
    print()


--- PitchCall (%) ---
PitchCall
BallCalled              35.60
InPlay                  18.26
FoulBallNotFieldable    16.39
StrikeCalled            15.40
StrikeSwinging          11.33
BallinDirt               1.79
FoulBall                 0.44
HitByPitch               0.36
FoulBallFieldable        0.30
Undefined                0.13
BallIntentional          0.00
AutomaticBall            0.00
AutomaticStrike          0.00
Name: proportion, dtype: float64

--- hit_type (%) ---
hit_type
NaN           71.02
GroundBall    10.06
FlyBall        7.93
LineDrive      5.95
Popup          4.68
Bunt           0.36
Name: proportion, dtype: float64

--- play_result (%) ---
play_result
BallCalled           33.19
NeutralPlay          31.37
Out                  10.54
StrikeSwinging        7.49
Strikeout             4.83
Single                4.50
Walk                  2.49
BallinDirt            1.71
Double                1.23
HomeRun               0.73
FieldersChoice        0.57
HitByPitch            0.36


/tmp/ipykernel_13815/4154581370.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Guardar resumen de hallazgos en `reports/eda_summary.md`

In [9]:
n_total = len(df)
n_missing_alt = int(df["altitude_category"].isna().sum())

harp_share = df["altitude_category"].value_counts(normalize=True, dropna=False).get(HARP_LABEL, 0) * 100
year_counts_clean = {int(k): int(v) for k, v in year_counts.items()}

# comparacion de medias Harp vs resto para InducedVertBreak / HorzBreak, por pitch type (top 3 mas comunes)
top_types = order[:4] if 'order' in dir() else pitchtype_counts.index[:4].tolist()
comparison_rows = []
for pt in top_types:
    sub = df_box_valid[df_box_valid["AutoPitchType"] == pt]
    harp = sub[sub["altitude_group"] == "Harp Helu (Extreme Altitude)"]
    resto = sub[sub["altitude_group"] == "Resto de estadios"]
    comparison_rows.append({
        "pitch_type": pt,
        "ivb_harp": harp["InducedVertBreak"].mean(),
        "ivb_resto": resto["InducedVertBreak"].mean(),
        "hb_harp": harp["HorzBreak"].abs().mean(),
        "hb_resto": resto["HorzBreak"].abs().mean(),
        "velo_harp": harp["RelSpeed"].mean(),
        "velo_resto": resto["RelSpeed"].mean(),
    })
comparison_df = pd.DataFrame(comparison_rows).round(2)

summary_lines = []
summary_lines.append("# EDA — Resumen de hallazgos (Stuff+ LMB)\n")
summary_lines.append(
    f"Dataset: `data/raw/stuff_model_df.parquet` — **{n_total:,} pitches**, "
    f"**{n_unique_pitchers} pitchers unicos**, temporadas {sorted(df['year'].unique().tolist())}.\n"
)

summary_lines.append("## 1. Cobertura y balance de datos\n")
summary_lines.append(
    f"- `altitude_category` tiene {n_missing_alt} valores nulos "
    f"({100*n_missing_alt/n_total:.2f}% del total), a excluir o imputar antes de modelar.\n"
    f"- **Extreme Altitude** (asumido Harp Helu) representa **{harp_share:.1f}%** de los pitches "
    f"totales — desbalance esperado dado que es 1 de N estadios, pero suficiente volumen "
    f"({int(alt_counts.get(HARP_LABEL, 0)):,} pitches) para modelar su efecto con confianza.\n"
    f"- Distribucion por year: {year_counts_clean}.\n"
    f"- `AutoPitchType` esta dominado por Four-Seam, Slider y Sinker; hay clases residuales "
    f"con muestra casi nula (OneSeamFastBall, TwoSeamFastBall, Knuckleball con 1 pitch c/u, "
    f"Sweeper con {int(pitchtype_counts.get('Sweeper', 0))}) que deben agruparse en 'Other' o "
    f"excluirse del modelo por falta de muestra.\n"
)

summary_lines.append("## 2. Estadisticas descriptivas\n")
summary_lines.append(
    "Ver tablas completas en `reports/desc_stats_by_altitude.csv` y "
    "`reports/desc_stats_by_pitchtype.csv` (mean, std, p1/p25/p50/p75/p99 de RelSpeed, "
    "SpinRate, InducedVertBreak, HorzBreak, Extension, RelHeight).\n"
)

summary_lines.append("## 3. Efecto de altitud sobre movimiento (Harp Helu vs resto)\n")
summary_lines.append(
    "Comparacion de medias por tipo de pitch (grupos con n>=200 en ambos lados):\n\n"
    + comparison_df.to_markdown(index=False) + "\n\n"
)
summary_lines.append(
    "- El patron esperado por fisica de la altitud (menor densidad de aire -> menos "
    "movimiento vertical inducido y menos movimiento horizontal, mayor velocidad efectiva) "
    "debe leerse directamente de la tabla anterior antes de asumir causalidad: esto es "
    "correlacional, confunde altitud con plantilla de pitchers que juegan en Harp Helu, y "
    "requiere el modelado causal (PyMC/econml) planeado en fases posteriores.\n"
    "- Ver boxplots `reports/figures/02_box_inducedvertbreak_harp_vs_resto.png` y "
    "`03_box_horzbreak_harp_vs_resto.png` para la distribucion completa por pitch type.\n"
)

summary_lines.append("## 4. Muestra por pitcher (shrinkage bayesiano)\n")
summary_lines.append(
    f"- {n_unique_pitchers} pitchers unicos, con una distribucion de pitches por pitcher muy "
    f"sesgada a la derecha: mediana de {pitcher_desc['50%']:.0f} pitches, pero p10={pitcher_desc['10%']:.0f} "
    f"y p1={pitcher_desc['1%']:.0f}, contra un maximo de {pitcher_desc['max']:.0f}.\n"
    f"- **{n_low_sample} pitchers ({pct_low_sample:.1f}%) tienen menos de 100 pitches registrados.** "
    f"Esto confirma la necesidad de shrinkage bayesiano (partial pooling) hacia la media de la "
    f"liga/tipo de pitch para evitar sobreajustar las metricas de pitchers con poca muestra.\n"
    f"- Ver histograma en `reports/figures/04_pitches_por_pitcher_hist.png`.\n"
)

summary_lines.append("## 5. Distribucion de resultados del pitch\n")
summary_lines.append(
    "Ver `reports/figures/05_pct_pitchcall_hittype_playresult.png` y tablas impresas en el "
    "notebook. Puntos relevantes:\n"
    f"- `hit_type` es NaN en {pct_tables['hit_type'].get('NaN', 0):.1f}% de los casos "
    "(pitches que no resultan en bola en juego bateada), por lo que solo tiene sentido "
    "analizarlo condicionado a `is_ball_in_play`/`is_batted`.\n"
    f"- `PitchCall` esta dominado por BallCalled ({pct_tables['PitchCall'].get('BallCalled', 0):.1f}%), "
    f"InPlay ({pct_tables['PitchCall'].get('InPlay', 0):.1f}%) y "
    f"FoulBallNotFieldable ({pct_tables['PitchCall'].get('FoulBallNotFieldable', 0):.1f}%).\n"
    f"- Eventos raros como HomeRun, Triple y Error en `play_result` tienen muestra muy baja "
    "y deben tratarse con cuidado en cualquier modelo run-value-based.\n"
)

summary_lines.append("## Siguientes pasos sugeridos\n")
summary_lines.append(
    "- Confirmar con Diablos que `Extreme Altitude` == Harp Helu (asuncion usada en este EDA).\n"
    "- Definir tratamiento de `altitude_category` NaN antes de features.\n"
    "- Excluir o agrupar `AutoPitchType` con muestra insuficiente (<50 pitches).\n"
    "- Diseñar el nivel de partial pooling (por pitcher x pitch type) para el modelo bayesiano, "
    "dado el fuerte sesgo en pitches-por-pitcher.\n"
)

with open(REPORT_DIR / "eda_summary.md", "w") as f:
    f.write("\n".join(summary_lines))

print("Guardado: reports/eda_summary.md")


Guardado: reports/eda_summary.md
